# 02 - Tools and MCP

**Case study:** Maple quotes an indicative rate, pre-qualifies a borrower, then escalates a self-employed edge case, with a compliance approval gate.

## What you'll learn
- Function tools (`@tool`) and how to see every tool call
- `approval_mode="always_require"` for sensitive actions (human-in-the-loop)
- MCP: a remote server (Microsoft Learn) and a local server we write (stdio)
- Foundry-side MCP tool on a prompt agent, and the Foundry tool catalog / toolbox

## 1. Setup

In [1]:
import sys, pathlib, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, str(pathlib.Path.cwd().parent))   # make `shared` importable (notebook cwd = notebooks/)
from shared import fsi_common as fc
from shared.fsi_common import *

settings = load_settings(require=("project_endpoint", "model"))
print_env_check(settings)

AZURE_SUBSCRIPTION_ID                      set
AZURE_RESOURCE_GROUP                       set
AZURE_LOCATION                             set
AZURE_AI_PROJECT_ENDPOINT                  set
AZURE_AI_MODEL_DEPLOYMENT_NAME             set
AZURE_AI_EMBEDDING_DEPLOYMENT_NAME         set
AZURE_SEARCH_ENDPOINT                      set
AZURE_SEARCH_INDEX_NAME                    set
AZURE_SEARCH_KNOWLEDGE_BASE_NAME           set
AZURE_AI_KB_CONNECTION_NAME                set
APPLICATIONINSIGHTS_CONNECTION_STRING      set
AZURE_CONTAINER_REGISTRY_NAME              set


In [2]:
def show_calls(resp):
    """Tool-call visibility: print every call and result in a response."""
    for m in resp.messages:
        for c in m.contents:
            if c.type in ("function_call", "mcp_server_tool_call"):
                print(f"  CALL   {getattr(c, 'tool_name', None) or c.name}({redact(str(c.arguments))})")
            elif c.type in ("function_result", "mcp_server_tool_result"):
                print(f"  RESULT {redact(str(c.result))[:160]}")

## 2. Function tools
`fc.TOOLS` wraps five plain Python functions (`get_rate_quote`, `prequalify`, `get_document_checklist`, `lookup_customer`, `escalate_to_underwriter`) with `@tool`. The model reads each function's name, type hints and docstring to decide when to call it. Data is fictional and deterministic.

In [3]:
import inspect
for fn in (get_rate_quote, prequalify, get_document_checklist):
    print(fn.__name__, inspect.signature(fn))

get_rate_quote (term_years: 'int', credit_band: 'str', ltv_pct: 'float') -> 'dict'
prequalify (annual_income: 'float', monthly_debts: 'float', down_payment: 'float', price: 'float') -> 'dict'
get_document_checklist (employment_type: 'str') -> 'list[str]'


In [4]:
agent = make_agent("maple-tools", MAPLE_INSTRUCTIONS, tools=fc._build_tools("never_require")[:4])
q = ("Borrower: 120000 income, 600 monthly debts, buying a 500000 home with 100000 down, good credit. "
     "Pre-qualify them and quote an indicative 5-year rate.")
resp = await agent.run(q)
show_calls(resp)
show(resp, title="Answer")

  CALL   prequalify({"annual_income":120000,"down_payment":100000,"monthly_debts":600,"price":500000})
  CALL   get_rate_quote({"credit_band":"good","ltv_pct":80,"term_years":5})
  RESULT {"gds_pct": 30.8, "tds_pct": 36.8, "ltv_pct": 80.0, "decision": "likely", "reasons": ["Within guideline limits"], "disclosure": "Indicative only. Not a commitme
  RESULT {"rate_pct": 4.79, "apr_pct": 4.86, "term_years": 5, "as_of": "2026-11-01", "indicative": true, "disclosure": "Indicative only. Not a commitment to lend. Rates 
=== Answer ===
The borrower is likely pre-qualified for the mortgage based on their income, debts, down payment, and home price. The loan-to-value ratio is 80%, with GDS and TDS within guideline limits.

An indicative 5-year fixed rate quote is 4.79% with an APR of 4.86%. 

Indicative only. Not a commitment to lend. Rates and eligibility are subject to full application, credit adjudication and underwriter approval.
[usage] {'openai.cache_write_tokens': 0, 'input_token_count': 

## 3. Approval gate on a sensitive tool
Escalating to an underwriter creates an audit record. Mark it `approval_mode="always_require"`: the run pauses and returns an approval *request*; a human approves or denies, and we resume in the same session.

In [5]:
from agent_framework import Message

gated = make_agent("maple-gated", MAPLE_INSTRUCTIONS, tools=fc._build_tools("always_require"))
session = gated.create_session()
q = "Case C-1002 is self-employed with variable income and a large gift. Please escalate to the underwriter."
resp = await gated.run(q, session=session)
for req in resp.user_input_requests:
    print("APPROVAL REQUESTED:", req.function_call.name, redact(str(req.function_call.arguments)))
print("audit log before approval:", fc.audit_log())

APPROVAL REQUESTED: escalate_to_underwriter {"case_id":"C-1002","reason":"Self-employed with variable income and a large gift, requires human underwriter review."}
audit log before approval: []


In [6]:
APPROVE = True   # change to False to see a denial
msgs = [q, Message("assistant", resp.user_input_requests)]
msgs += [Message("user", [r.to_function_approval_response(APPROVE)]) for r in resp.user_input_requests]
resp2 = await gated.run(msgs, session=session)
show(resp2, title="After human decision")
print("audit log after:", fc.audit_log())

=== After human decision ===
Case C-1002 has been escalated to a human underwriter for review due to self-employment with variable income and a large gift. A specialist will assess the details shortly.
[usage] {'openai.cache_write_tokens': 0, 'input_token_count': 585, 'openai.cached_input_tokens': 0, 'total_token_count': 624, 'openai.reasoning_tokens': 0, 'output_token_count': 39, 'reasoning_output_token_count': 0, 'cache_read_input_token_count': 0, 'cache_creation_input_token_count': 0}
audit log after: [{'ticket_id': 'UW-09427', 'case_id': 'C-1002', 'reason': 'Self-employed with variable income and a large gift, requires human underwriter review.'}]


## 4. Remote MCP: Microsoft Learn
[MCP](https://modelcontextprotocol.io) standardizes how agents discover and call tools. `MCPStreamableHTTPTool` connects to a remote server; its tools are discovered at runtime. Useful for FSI teams grounding answers in vendor/regulatory documentation.

In [7]:
from agent_framework import MCPStreamableHTTPTool

async with MCPStreamableHTTPTool("learn", "https://learn.microsoft.com/api/mcp") as learn:
    print("discovered tools:", [t.name for t in learn.functions])
    a = make_agent("maple-learn", "Answer briefly, using the Learn tools for Azure facts. Cite the URL.", tools=[learn])
    r = await a.run("According to Microsoft Learn, what is a Foundry hosted agent? Two sentences.")
    show_calls(r)
    show(r, title="Grounded answer")

discovered tools: ['microsoft_docs_search', 'microsoft_code_sample_search', 'microsoft_docs_fetch']


  CALL   microsoft_docs_search({"query":"Foundry hosted agent"})
  RESULT {"results": [{"title": "What are hosted agents?", "content": "# What are hosted agents?\nWhen you build agentic applications by using open-source frameworks, yo
=== Grounded answer ===
A Foundry hosted agent in Microsoft Foundry Agent Service is a containerized agent application deployed to Microsoft-managed infrastructure that handles scaling, session state persistence, security, and lifecycle management, allowing you to focus on your agent's logic. It allows custom agent code or preferred agent frameworks to run securely with dedicated identity and OpenAI-compatible endpoints, supporting stateful and long-lived workloads. 

Source: https://learn.microsoft.com/azure/foundry/agents/concepts/hosted-agents
[usage] {'openai.cache_write_tokens': 0, 'input_token_count': 6150, 'openai.cached_input_tokens': 5376, 'total_token_count': 6268, 'openai.reasoning_tokens': 0, 'output_token_count': 118, 'reasoning_output_token_

## 5. Local MCP server (stdio)
`shared/mcp_loan_server.py` is a ~20-line MCP server exposing two Northwind tools. The agent launches it as a subprocess and discovers the tools.

In [8]:
print(open(fc.DAY2_ROOT / "shared" / "mcp_loan_server.py").read())

"""Tiny local MCP server (stdio) exposing Maple mortgage helpers. Run: python mcp_loan_server.py"""
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path(__file__).resolve().parents[1]))
from mcp.server.mcpserver import MCPServer  # noqa: E402

from shared import fsi_common as f  # noqa: E402

mcp = MCPServer("northwind-loan-tools")


@mcp.tool()
def document_checklist(employment_type: str) -> list[str]:
    """Documents required for a mortgage application (salaried, self-employed, retired)."""
    return f.get_document_checklist(employment_type)


@mcp.tool()
def indicative_rate(term_years: int, credit_band: str, ltv_pct: float) -> dict:
    """Indicative (non-binding) fixed rate quote."""
    return f.get_rate_quote(term_years, credit_band, ltv_pct)


if __name__ == "__main__":
    mcp.run(transport="stdio")



In [9]:
import sys
from agent_framework import MCPStdioTool

server = fc.DAY2_ROOT / "shared" / "mcp_loan_server.py"
import os
from mcp.client.stdio import StdioServerParameters, stdio_client

class NotebookStdioTool(MCPStdioTool):
    """Jupyter's stderr has no fileno(), which the stdio client needs: send server stderr to a real file."""
    def get_mcp_client(self):
        params = StdioServerParameters(command=self.command, args=self.args, env=self.env)
        return stdio_client(params, errlog=open(os.devnull, "w"))

async with NotebookStdioTool("northwind-loan", sys.executable, args=[str(server)]) as loan:
    print("discovered tools:", [t.name for t in loan.functions])
    a = make_agent("maple-mcp", MAPLE_INSTRUCTIONS, tools=[loan])
    r = await a.run("What documents does a self-employed applicant need to provide?")
    show_calls(r)
    show(r, title="Answer via local MCP")

discovered tools: ['document_checklist', 'indicative_rate']


  CALL   document_checklist({"employment_type":"self-employed"})
  RESULT {"result": ["Government photo ID", "Two years' Notices of Assessment and T1 Generals", "Business registration or articles of incorporation", "Two years' financi
=== Answer via local MCP ===
A self-employed applicant needs to provide the following documents:
- Government photo ID
- Two years' Notices of Assessment and T1 Generals
- Business registration or articles of incorporation
- Two years' financial statements
- 90 days of business and personal bank statements
- Confirmation of the down-payment source

Let me know if you need details on any other employment type or further help!
[usage] {'openai.cache_write_tokens': 0, 'input_token_count': 650, 'openai.cached_input_tokens': 0, 'total_token_count': 749, 'openai.reasoning_tokens': 0, 'output_token_count': 99, 'reasoning_output_token_count': 0, 'cache_read_input_token_count': 0, 'cache_creation_input_token_count': 0}


## 6. Server-side MCP on a prompt agent
With a prompt agent, **Foundry** calls the MCP server for you (no client-side loop). Add an `MCPTool` to the `PromptAgentDefinition`. Only publicly reachable servers work; private ones need a project connection.

In [10]:
from azure.ai.projects.models import PromptAgentDefinition, MCPTool

pc = get_project_client(settings)
v = pc.agents.create_version(
    agent_name="maple-learn-mcp",
    definition=PromptAgentDefinition(
        model=settings.model,
        instructions="Answer briefly using Microsoft Learn. Cite the URL.",
        tools=[MCPTool(server_label="learn", server_url="https://learn.microsoft.com/api/mcp", require_approval="never")],
    ),
)
print("created", v.name, "version", v.version)
remote = get_foundry_agent("maple-learn-mcp")
r = await remote.run("Using Learn, what is a Foundry hosted agent? One sentence.")
show_calls(r)
show(r, title="Server-side MCP answer")

created maple-learn-mcp version 2


  CALL   microsoft_docs_search({"query":"Foundry hosted agent"})
  RESULT None
=== Server-side MCP answer ===
A Foundry hosted agent is a containerized AI agent deployed on Microsoft-managed infrastructure that handles scaling, session state persistence, security, and lifecycle management, allowing you to focus on your agent's logic while the platform manages the orchestration and calls to Foundry models and tools.  
Source: https://learn.microsoft.com/azure/foundry/agents/concepts/hosted-agents
[usage] {'openai.cache_write_tokens': 0, 'input_token_count': 10834, 'openai.cached_input_tokens': 0, 'total_token_count': 10934, 'openai.reasoning_tokens': 0, 'output_token_count': 100, 'reasoning_output_token_count': 0, 'cache_read_input_token_count': 0, 'cache_creation_input_token_count': 0}


## 7. Foundry tool catalog and toolbox
Foundry offers built-in tools (web search, file search, code interpreter, Azure AI Search, ...) and **toolboxes**: curated, centrally governed bundles of tools exposed through one MCP endpoint (`FoundryToolbox`). Let's see what exists in this project.

In [11]:
try:
    boxes = list(pc.toolboxes.list())
    print("toolboxes in project:", [getattr(b, "name", b) for b in boxes] or "none")
except Exception as e:
    print("toolboxes unavailable:", type(e).__name__, str(e)[:200])

# Built-in catalog tools are available as helpers on the chat client:
client = get_chat_client(settings)
print("catalog helpers:", sorted(n for n in dir(client) if n.startswith("get_") and n.endswith("_tool") and not n.startswith("_")))

toolboxes in project: none
catalog helpers: ['get_a2a_tool', 'get_azure_ai_search_tool', 'get_bing_custom_search_tool', 'get_bing_grounding_tool', 'get_browser_automation_tool', 'get_code_interpreter_tool', 'get_computer_tool', 'get_computer_use_tool', 'get_fabric_tool', 'get_file_search_tool', 'get_image_generation_tool', 'get_mcp_tool', 'get_memory_search_tool', 'get_sharepoint_tool', 'get_shell_tool', 'get_web_search_tool']


In [12]:
# Use a built-in catalog tool: code interpreter does the amortization maths with real code.
a = make_agent("maple-calc", "You are a careful mortgage calculator. Use code for arithmetic.", tools=[client.get_code_interpreter_tool()])
try:
    r = await a.run("Monthly payment on a 400000 loan at 4.79% over 25 years, monthly compounding? Give one number.")
    show(r, title="Code interpreter (catalog tool)")
except Exception as e:
    print("Code interpreter unavailable here:", type(e).__name__, str(e)[:300])

=== Code interpreter (catalog tool) ===
To calculate the monthly payment for a loan, we use the formula for the annuity payment:

\[
M = P \frac{r(1+r)^n}{(1+r)^n - 1}
\]

where:
- \(M\) is the monthly payment,
- \(P\) is the loan principal (400,000),
- \(r\) is the monthly interest rate (annual rate divided by 12),
- \(n\) is the total number of payments (years multiplied by 12).

Let's calculate it.The monthly payment on a $400,000 loan at 4.79% annual interest over 25 years with monthly compounding is approximately $2289.68.
[usage] {'openai.cache_write_tokens': 0, 'input_token_count': 335, 'openai.cached_input_tokens': 0, 'total_token_count': 595, 'openai.reasoning_tokens': 0, 'output_token_count': 260, 'reasoning_output_token_count': 0, 'cache_read_input_token_count': 0, 'cache_creation_input_token_count': 0}


> **Optional / preview:** toolboxes and some catalog tools (web search, Fabric, SharePoint) depend on region, tenant and preview features. If the project has no toolbox the listing above is empty; create one in the portal (Tools -> Toolboxes) and connect with `FoundryToolbox(credential, url=...)`. We do not fake output for unavailable capabilities.

## Key takeaways
- Plain Python functions become tools with `@tool`; type hints and docstrings are the contract.
- Always inspect tool calls (`show_calls`): they are your audit trail.
- `always_require` gives a human-in-the-loop gate for sensitive actions; resume with the same session.
- MCP = plug-and-play tools: remote (Learn), local stdio (ours), or server-side on a prompt agent.
- Toolboxes centralize governed tool access for many agents.

**Next up:** [03-foundry-iq-knowledge.ipynb](03-foundry-iq-knowledge.ipynb) - ground the agent in underwriting policy.